# Episode 26: Security

An agent that can call tools can also do damage — send the wrong payment, leak data, run a destructive command. Production agents need guard rails.

**In this episode you'll build:** two security layers — a **human approval gate** on sensitive tools, and a **tamper-evident audit trail** of everything the agent did.

## Defense in depth

No single mechanism makes an agent safe. You layer them:

| Layer | Mechanism | Episode |
|---|---|---|
| Observe | Telemetry — see what happened | 24 |
| Intervene | Observers — block dangerous actions automatically | 25 |
| **Approve** | **HITL gates — a human signs off on sensitive calls** | **26** |
| **Audit** | **A durable log of every tool call and result** | **26** |

Episodes 24 and 25 covered the first two. This episode adds the human and the paper trail.

## Layer 1: human approval gates

Some tool calls are too consequential to let an agent make alone — payments, deletions, emails. The `approval_required()` tool middleware **pauses** before such a tool runs and asks a human.

The human's answer comes from the agent's `hitl_hook` — a callback that receives the approval prompt and returns a decision. In a real app the hook shows a UI dialog; here it applies a **policy**: approve payments at or under \$100, deny anything larger.

In [1]:
from dotenv import load_dotenv

load_dotenv()

import re

from autogen.beta import Agent, tool
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import HumanInputRequest, HumanMessage
from autogen.beta.middleware import approval_required

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

APPROVAL_LIMIT = 100.0


@tool(middleware=[approval_required()])
def send_payment(to: str, amount: float) -> str:
    """Send a payment to a recipient."""
    return f"Paid ${amount:.0f} to {to}."


def policy_hitl(event: HumanInputRequest) -> HumanMessage:
    """Approve payments <= $100, deny the rest. A real hook would show a UI."""
    match = re.search(r'"amount":\s*([0-9.]+)', event.content)
    amount = float(match.group(1)) if match else 0.0
    approved = amount <= APPROVAL_LIMIT
    print(f"[approval]  ${amount:.0f}  ->  {'APPROVE' if approved else 'DENY'}")
    return HumanMessage(content="y" if approved else "n")


agent = Agent(
    "treasurer",
    prompt="Use send_payment to fulfil payment requests. Report the tool result.",
    config=config,
    tools=[send_payment],
    hitl_hook=policy_hitl,
)

small = await agent.ask("Send a payment of $50 to alice.")
print("small:", small.body)

large = await agent.ask("Send a payment of $5000 to bob.")
print("large:", large.body)

[approval]  $50  ->  APPROVE


small: The payment of $50 to alice has been sent. Is there anything else you would like to do?


[approval]  $5000  ->  DENY


large: It seems that the payment request was denied. Could you please confirm if you want me to proceed with sending the payment of $5000 to Bob?


## Layer 2: a tamper-evident audit trail

When something goes wrong in production, you need to answer *exactly what did the agent do?* Subscribe to the event stream and record every tool call and result into a structured log.

Because the log is built from the same event stream the agent runs on, it can't drift from reality — it **is** reality. Persist it to a database or append-only file and you have an audit trail.

In [2]:
from autogen.beta import MemoryStream
from autogen.beta.events import ToolCallEvent, ToolResultEvent


def get_balance(account: str) -> str:
    """Get the balance of an account."""
    return f"{account}: $1,250.00"


audit_log: list[str] = []
stream = MemoryStream()


@stream.where(ToolCallEvent | ToolResultEvent).subscribe()
def record(event) -> None:
    if isinstance(event, ToolCallEvent):
        audit_log.append(f"CALL    {event.name} {event.arguments}")
    else:
        text = event.result.parts[0].content
        audit_log.append(f"RESULT  {event.name} -> {text}")


banker = Agent(
    "banker",
    prompt="Use get_balance to answer balance questions.",
    config=config,
    tools=[get_balance],
)

reply = await banker.ask("What's the balance of account ACME-01?", stream=stream)
print("reply:", reply.body)
print("--- audit log ---")
for i, line in enumerate(audit_log, 1):
    print(f"{i}. {line}")

reply: The balance of account ACME-01 is $1,250.00.
--- audit log ---
1. CALL    get_balance {"account":"ACME-01"}
2. RESULT  get_balance -> ACME-01: $1,250.00


## Combining the layers

A production agent uses all of these at once:

```python
agent = Agent(
    "production_agent",
    prompt="...",
    config=config,
    tools=[send_payment],                       # gated tool
    hitl_hook=policy_hitl,                       # Layer 3: approval
    observers=[PathGuardian(), TokenMonitor()],  # Layer 2: auto-intervention
    middleware=[TelemetryMiddleware(...)],       # Layer 1: telemetry
)
# ...and an audit subscriber on the stream passed to ask().
```

None of the layers know about each other — each is independent and composable. Add or remove one without touching the rest.

## Additional content

**Approval prompt customization.** `approval_required(message=..., denied_message=...)` tailors what the human sees and what the agent is told on a denial — so the agent can react gracefully instead of just failing.

**HITL is not just approval.** `context.input(...)` inside a tool can ask the human for *anything* — a missing API key, a clarification, a choice between options. Approval is the most common case, not the only one.

**Audit beyond tools.** The stream carries `ModelRequest`, `ModelResponse`, `ObserverAlert`, `HaltEvent` and more. A thorough audit log records those too — the full decision trail, not just the tool calls.

## What's next

Security keeps a production agent safe. Episode 27 keeps it *correct* — **testing** agents deterministically, with no API calls and no cost.